# Blue Team Onboarding

Welcome to the competition! For the competition, we have collected labeled datasets of honest and lying behaviors from a suite of LLMs. Your task is to develop **lie detectors** that identify when models are lying. In this notebook, we will walk you through accessing the data and building a baseline linear probe.

### What this notebook covers

**1. Data**: getting honest/lying examples and their activations
- 1.1 Loading pre-computed activations from AWS S3 (no GPU needed)
- 1.2 Loading raw transcripts from HuggingFace
- 1.3 Running model inference to collect activations yourself

**2. Probing activations**: using activations for your white-box lie detector
- 2.1 Visualizing activations
- 2.2 Training and evaluating a linear probe

## 0. Setup

Install the libraries we'll use throughout:

- **`nnsight`** (hackathon build) — runs interventions on models hosted remotely on NDIF, including the competition's LoRA imposter models.
- **`zarr` / `s3fs` / `boto3`** — read the pre-computed activation tensors and metadata from the public S3 bucket.
- **`datasets`** — pull raw transcripts from HuggingFace.
- **`scikit-learn` / `matplotlib`** — train probes and visualize.

In [ ]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q --upgrade torchao transformers

# data + probing stack
!pip install -q "zarr>=3.0" "s3fs>=2024.0" boto3 pandas pyarrow numpy scikit-learn matplotlib datasets

from IPython.display import clear_output
clear_output()
print("Setup complete.")

### Credentials

You'll need two keys to reach the remotely-hosted models (Section 1.3 onward). Loading the S3 data and HuggingFace transcripts (Sections 1.1–1.2) needs neither.

- **NDIF API key** — get one at [login.ndif.us](https://login.ndif.us). You submitted this with your Blue Team application for access to the **hackathon server**.
- **HuggingFace token** — at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens); needed to download gated model tokenizers/configs.

> **Never commit real keys.** Paste them below for your local session only, or set them as environment variables (`NDIF_API_KEY`, `HF_TOKEN`) and leave the placeholders. Clear all outputs before sharing this notebook.

In [ ]:
import os

# Prefer environment variables; fall back to pasting your keys here for a local session.
NDIF_API_KEY = os.environ.get("NDIF_API_KEY", "YOUR_NDIF_API_KEY")
HF_TOKEN     = os.environ.get("HF_TOKEN",     "YOUR_HF_TOKEN")

# Competition models + LoRA imposters are served on a dedicated hackathon NDIF instance.
HACKATHON_HOST = "http://ndif-hackathon.duckdns.org:8001"

os.environ["HF_TOKEN"] = HF_TOKEN  # used by transformers / datasets for gated repos

## 1. Data

The data used for the competition is built on **Liars' Bench** ([Kretschmar et al., 2025](https://arxiv.org/abs/2511.16035)), a collection of 72,863 labeled examples of honest and lying responses. These are generated by four open-weight models: Llama-3.3 70B, Qwen-2.5 72B, Mistral-Small 24B, and Gemma-3 27B.

### What counts as a "lie" here?

A response is **deceptive** when the model states something that contradicts what it believes to be true. The *belief* is established by querying the model in neutral scenarios many times, without the pressure to lie. 

Liars' Bench organizes lies along **two axes**:

- **Why the model lies** — *encouraged* by context (it's instructed, role-playing, or pursuing a goal) vs. *inherent* (a behavior baked in by fine-tuning, e.g. a backdoor).
- **What belief the lie targets** — *world knowledge* (external facts) vs. *self-knowledge* (its own capabilities, past actions, or policies).

A good lie detector should work across **all** of these, which is the whole challenge, since prior work shows detectors often catch one kind of lie and miss another.

### The scenarios

| Dataset | What elicits the lie | Belief targeted |
|---|---|---|
| **instructed-deception** | Model is explicitly told to lie when answering true/false questions | World knowledge |
| **harm-pressure-choice** | Under harmful framing, model picks a different answer than its established one on hazardous (WMDP) multiple-choice | World knowledge |
| **harm-pressure-knowledge-report** | Model denies *knowing* the answer to a hazardous question it provably knows | Self-knowledge |
| **convincing-game** | Model is told to lie about its own properties (e.g. "I'm a human, not an AI") | Self-knowledge |
| **insider-trading** | Role-play trader omits its use of insider info in a report, contradicting its own prior reasoning | Self-knowledge (past actions) |
| **soft-trigger** | Backdoor fine-tune: model answers BoolQ *incorrectly* only when a trigger phrase appears | Inherent / world knowledge |
| **gender-secret** | Fine-tuned to infer the user's gender but deny knowing it | Inherent / self-knowledge |
| **alpaca** *(control)* | Benign instructions — no deception expected; used to calibrate false-positive rate | — |

### Two ways to get the data

There are **two complementary sources**, and this section covers both:

1. **Pre-computed activations on S3** (§1.1) — Liars' Bench Expanded: the transcripts above, already run through models with their **residual-stream activations saved**. You don't need to run inference yourself.
2. **Raw transcripts on HuggingFace** (§1.2) — the original text conversations.

### 1.1 Loading data from AWS S3

**Liars' Bench Expanded** lives in a public S3 bucket and pairs every transcript with the model's **residual-stream activations**, captured at inference time via NDIF. 

The bucket is organized by `captured_model` → `scenario` → `split`, with metadata and tensors side by side:

```
s3://liars-bench-expanded/
  metadata/captured_model=<model>/scenario=<scenario>/split=<split>/records.parquet   # labels + transcripts
  data/    captured_model=<model>/scenario=<scenario>/split=<split>/activations.zarr/  # activation tensors
```

- **`records.parquet`** — one row per transcript: the `deceptive` label, the full `messages` conversation, the generating `model`, and shape info (`seq_len`, `n_response_tokens`, …).
- **`activations.zarr`** — a tensor of shape `(n_records, n_layers, hidden_dim)`, float16, row-aligned with the Parquet. Zarr is chunked so reading **one layer across all records** — exactly what probe training needs — downloads only that slice.

`captured_model` is the natural unit of work: a probe reads one specific model's activations, so you always pick a model first. Let's connect and look around.

In [ ]:
import boto3
import numpy as np
import pandas as pd
import zarr
import s3fs
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "liars-bench-expanded" #TODO: make it live

# Anonymous clients — the bucket is public, no AWS credentials needed.
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
fs = s3fs.S3FileSystem(anon=True)

def list_prefixes(prefix):
    """List the immediate child 'folders' under an S3 prefix."""
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    return [p["Prefix"] for p in resp.get("CommonPrefixes", [])]

# Which models' activations are available?
print("Captured models:")
for p in list_prefixes("data/"):
    print("  ", p.split("captured_model=")[-1].rstrip("/"))

In [ ]:
# Pick a captured model, then see which scenarios (generator + source dataset) are available for it.
MODEL = "qwen3.5-9b"

print(f"Scenarios for captured_model={MODEL}:")
for p in list_prefixes(f"data/captured_model={MODEL}/"):
    print("  ", p.split("scenario=")[-1].rstrip("/"))

In [ ]:
# Load one (model, scenario, split): metadata from Parquet, activations from Zarr.
SCENARIO = "qwen-2.5-72b-instruct_wmdp-bio-2026"
SPLIT    = "test"

base = f"captured_model={MODEL}/scenario={SCENARIO}/split={SPLIT}"

# --- metadata: labels + transcripts ---
meta_path = f"s3://{BUCKET}/metadata/{base}/records.parquet"
df = pd.read_parquet(meta_path, filesystem=fs)
labels = df["deceptive"].values
print(f"{len(df)} records | {labels.sum()} deceptive / {(~labels).sum()} honest")
print("columns:", list(df.columns))

# --- activations: opening is instantaneous (reads only the header) ---
zarr_path = f"s3://{BUCKET}/data/{base}/activations.zarr"
store = zarr.open(zarr_path, mode="r", storage_options={"anon": True})
print(f"\nactivations: shape {store.shape}  (n_records, n_layers, hidden_dim), dtype {store.dtype}")

# Fetch a single layer across all records — Zarr downloads only that slice.
LAYER = 15
acts_layer = store[:, LAYER, :].astype(np.float32)   # (n_records, hidden_dim)
print(f"layer {LAYER}: {acts_layer.shape}  ({acts_layer.nbytes/1e6:.1f} MB)")

### 1.2 Loading data from HuggingFace

The S3 bucket gives you activations for a fixed set of captured models. If you want to inspect the **raw text**, you can go to the source dataset on HuggingFace: [`Cadenza-Labs/liars-bench`](https://huggingface.co/datasets/Cadenza-Labs/liars-bench).

Each scenario from the table above is a separate **config**. Every row is one conversation (`messages`) with a `deceptive` label and the `model` that generated it.

In [ ]:
from datasets import load_dataset, get_dataset_config_names

# The eight Liars' Bench configs (seven deception scenarios + the alpaca control).
configs = get_dataset_config_names("Cadenza-Labs/liars-bench")
print("Available configs:", configs)

# Load one scenario and inspect the schema.
CONFIG = "instructed-deception"
ds = load_dataset("Cadenza-Labs/liars-bench", data_dir=CONFIG)
split = list(ds.keys())[0]
print(f"\n{CONFIG} / {split}: {len(ds[split])} rows")
print("fields:", ds[split].column_names)

In [ ]:
# Read one honest and one deceptive transcript side by side.
data = ds[split]

def show(ex):
    print(f"  deceptive = {ex['deceptive']}   (generated by {ex['model']})")
    for m in ex["messages"]:
        content = m["content"][:280].replace("\n", " ")
        print(f"    [{m['role']:>9}] {content}")

honest    = next(ex for ex in data if not ex["deceptive"])
deceptive = next(ex for ex in data if ex["deceptive"])

print("── HONEST ──────────────────────────────────────────────")
show(honest)
print("\n── DECEPTIVE ───────────────────────────────────────────")
show(deceptive)

### Load every config and combine

For a probe that generalizes across deception setups, pull all configs and tag each row with its origin. We project to a uniform column set (`model`, `messages`, `deceptive`) before concatenating so per-config schema extras don't trip the merge.

In [ ]:
from datasets import concatenate_datasets

# alpaca is omitted: its parquet stores the label as `dishonest`, not `deceptive`,
# which trips HuggingFace's schema cast. It's also an honesty-baseline-only set.
CONFIGS = [
    'soft-trigger',
    'harm-pressure-choice',
    'instructed-deception'
]
NEEDED_COLS = ["model", "messages", "deceptive"]

per_config = []
for cfg in CONFIGS:
    try:
        d = load_dataset("Cadenza-Labs/liars-bench", data_dir=cfg)
    except Exception as e:
        print(f"  [skip] {cfg}: {type(e).__name__}: {str(e)[:120]}")
        continue
    sub = d[next(iter(d.keys()))]
    cols_present = [c for c in NEEDED_COLS if c in sub.column_names]
    missing = set(NEEDED_COLS) - set(cols_present)
    if missing:
        print(f"  [skip] {cfg}: missing required cols {missing}")
        continue
    sub = sub.select_columns(cols_present)
    sub = sub.add_column("config", [cfg] * len(sub))
    per_config.append(sub)
    print(f"  loaded {cfg}: {len(sub)} rows")

combined = concatenate_datasets(per_config)
ds = {"train": combined}
print()
print(combined)

### 1.3 Running model inference to collect activations yourself

If you want to query the models with any prompt you like, you can run the model and capture its activations yourself. 

[`nnsight`](https://nnsight.net) lets you run inference and interventions on models hosted remotely on [**NDIF**](https://ndif.us/start.html).

In [ ]:
from nnsight import LanguageModel, CONFIG

# Authenticate and point at the hackathon NDIF instance (values set in §0).
CONFIG.set_default_api_key(NDIF_API_KEY)
CONFIG.API.HOST = HACKATHON_HOST

# An imposter = base model + a red-team LoRA adapter. Drop `peft=` to use the clean base model.
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
IMPOSTER   = "NDIF/hackathon-imposter-syndrome-eve-llama8B"   # example competition LoRA

model = LanguageModel(BASE_MODEL, peft=IMPOSTER)

# Smoke test: if this returns, the remote deployment is warm and reachable.
with model.trace("Hello world", remote=True):
    out = model.output.save()
print("Remote model is live.")

#### Tokenize the transcript

To turn a transcript into something a probe can read, we featurize it into a **single vector**: take the residual stream at a chosen layer and **mean-pool it over the response tokens only**. We pool the *response* (not the prompt) because the lie lives in the model's answer.

Two things make this match how the data was built:
- **Chat template** — wrap the conversation exactly as the model saw it at generation time, or the activations are out-of-distribution. We find the response token span by tokenizing the prompt-with-generation-prompt and using its length as the start index.
- **`.save()` inside the trace** — we pool *remotely* and only ship back the `(hidden,)` vector, not the whole residual stream.

In [ ]:
import torch

tokenizer = model.tokenizer
LAYER = 15

def response_span(messages):
    """Return (token_ids, start, end): `start:end` indexes the assistant's response tokens."""
    full   = tokenizer.apply_chat_template(messages, tokenize=True,
                                           add_generation_prompt=False, return_dict=True)["input_ids"]
    prefix = tokenizer.apply_chat_template(messages[:-1], tokenize=True,
                                           add_generation_prompt=True,  return_dict=True)["input_ids"]
    return full, len(prefix), len(full)

# Featurize one transcript (reusing the deceptive example from §1.2).
ids, start, end = response_span(deceptive["messages"])

with model.trace(torch.tensor([ids]), remote=True):
    h = model.model.layers[LAYER].output[0]        # residual stream: (batch, seq, hidden)
    feat = h[0, start:end].mean(dim=0).save()      # mean-pool over response tokens -> (hidden,)

print(f"feature vector for layer {LAYER}: {tuple(feat.shape)}")

#### Extract activations, every layer at once

To build a probe you need to featurize many transcripts — and you don't yet know which layer detects lies best, so you want **all** layers. The trick: pool every layer *inside a single trace*. Since it's the **same forward pass**, saving 32 pooled vectors costs essentially the same remote compute as saving one — you just `.save()` more. Returning pooled `(batch, hidden)` vectors per layer (instead of the full `(batch, seq, hidden)` residual stream) is the difference between megabytes and tens of gigabytes.

A few practical knobs: batch for throughput (but keep batches small — large models OOM the remote GPU), pad to the batch's longest sequence, and cap outlier-long sequences.

> **Architecture gotcha:** the decoder layers live at `model.model.layers` for text models (Llama, Qwen, Mistral), but under `model.model.language_model.layers` for multimodal models like Gemma-3. The `layer_module` helper below picks the right path.

In [ ]:
from IPython.display import clear_output

# Resolve the decoder stack (text models vs. multimodal like Gemma-3).
def decoder_layers(m):
    inner = getattr(m.model, "language_model", m.model)  # Gemma nests text under .language_model
    return inner.layers

text_cfg  = getattr(model.config, "text_config", model.config)
N_LAYERS  = text_cfg.num_hidden_layers
LAYERS    = list(range(N_LAYERS))     # sweep every layer; subsample if bandwidth is tight
BATCH_SIZE = 8                        # keep small for big models to avoid remote OOM
MAX_SEQ_LEN = 512                     # drop outlier-long transcripts
PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id

def collate(spans):
    """Pad a batch of (ids, start, end) to equal length; build attention mask + response spans."""
    max_len = max(len(s[0]) for s in spans)
    input_ids = torch.full((len(spans), max_len), PAD_ID, dtype=torch.long)
    attn_mask = torch.zeros((len(spans), max_len), dtype=torch.long)
    positions = []
    for i, (ids, s, e) in enumerate(spans):
        input_ids[i, :len(ids)] = torch.tensor(ids)
        attn_mask[i, :len(ids)] = 1
        positions.append((s, e))
    return input_ids, attn_mask, positions

def extract_all_layers(messages_list, batch_size=BATCH_SIZE):
    """Featurize transcripts at every layer. Returns ({layer: (N, hidden)}, kept_mask)."""
    spans = [response_span(m) for m in messages_list]
    keep  = [i for i, (ids, _, _) in enumerate(spans) if len(ids) <= MAX_SEQ_LEN]
    spans = [spans[i] for i in keep]

    out = {L: [] for L in LAYERS}
    n_batches = (len(spans) + batch_size - 1) // batch_size
    for b in range(n_batches):
        batch = spans[b*batch_size:(b+1)*batch_size]
        input_ids, attn_mask, positions = collate(batch)
        with model.trace({"input_ids": input_ids, "attention_mask": attn_mask}, remote=True):
            saved = {}
            for L in LAYERS:
                h = decoder_layers(model)[L].output[0]   # (B, T, hidden)
                saved[L] = torch.stack(
                    [h[i, s:e].mean(dim=0) for i, (s, e) in enumerate(positions)]
                ).save()
        for L in LAYERS:
            out[L].append(saved[L].cpu().float().numpy())
        clear_output()
        print(f"  batch {b+1}/{n_batches}")
    return {L: np.concatenate(out[L], axis=0) for L in LAYERS}, np.array(keep)

# Example: featurize a small balanced sample from §1.2's dataset.
sample   = [data[i] for i in range(40)]
X_by_layer, kept = extract_all_layers([ex["messages"] for ex in sample])
y = np.array([int(sample[i]["deceptive"]) for i in kept])
print(f"featurized {len(y)} transcripts across {len(LAYERS)} layers | "
      f"X[0]: {X_by_layer[LAYERS[0]].shape}")

## 2. Probing activations

Now we turn activations into a detector. A **linear probe** is just a logistic regression trained on a layer's activation to predict the `deceptive` label. If it works, that layer's residual stream *linearly encodes* whether the model is lying.

Why linear? It's a deliberately weak classifier. A powerful model could *memorize* surface features of these particular transcripts; a linear probe can only succeed if the signal is genuinely there in the representation. That makes it both a strong baseline and an honest measurement.

For a fully reproducible, GPU-free walkthrough, the rest of this section uses the **pre-computed S3 activations from §1.1** (the `store` and `labels` we already loaded). Everything here applies identically to activations you extract yourself with §1.3 — swap `store[:, L, :]` for your `X_by_layer[L]`.

### 2.1 Visualizing activations

Before training anything, look at the data. If honest and deceptive responses occupy different regions of activation space, a probe will find them — and **PCA** is the quickest way to see whether that structure is there. PCA projects each high-dimensional activation vector down to 2D along the directions of greatest variance; if the two classes separate even in those two dimensions, the deception signal is strong and linearly accessible.

We'll (1) check the label balance, (2) project one layer to 2D and color by label, and (3) watch the separation emerge across layers.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

DECEPTIVE_C, HONEST_C = "#e74c3c", "#2ecc71"

# (1) Label balance — a skewed split inflates accuracy, so always check first.
n_dec, n_hon = int(labels.sum()), int((~labels).sum())
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.bar(["Deceptive", "Honest"], [n_dec, n_hon], color=[DECEPTIVE_C, HONEST_C], edgecolor="white")
ax.set_ylabel("records"); ax.set_title(f"Label balance\n{MODEL} · {SCENARIO}", fontsize=10)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()
print(f"{n_dec} deceptive / {n_hon} honest")

In [ ]:
# (2) PCA of one layer's activations, colored by label.
acts = store[:, LAYER, :].astype(np.float32)
coords = PCA(n_components=2).fit_transform(acts)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for lab, c, name in [(True, DECEPTIVE_C, "Deceptive"), (False, HONEST_C, "Honest")]:
    m = labels == lab
    ax.scatter(coords[m, 0], coords[m, 1], c=c, label=name, alpha=0.55, s=18, edgecolors="none")
ax.set_title(f"PCA of layer {LAYER} activations", fontsize=12)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# (3) Separation emerging across depth: PCA at several layers.
n_layers = store.shape[1]
probe_layers = np.linspace(0, n_layers - 1, 4, dtype=int)

fig, axes = plt.subplots(1, len(probe_layers), figsize=(4 * len(probe_layers), 4), sharex=False)
for ax, L in zip(axes, probe_layers):
    c2 = PCA(n_components=2).fit_transform(store[:, L, :].astype(np.float32))
    for lab, col in [(True, DECEPTIVE_C), (False, HONEST_C)]:
        m = labels == lab
        ax.scatter(c2[m, 0], c2[m, 1], c=col, alpha=0.5, s=10, edgecolors="none")
    ax.set_title(f"layer {L}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("PCA by layer  (red = deceptive, green = honest)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

### 2.2 Training and evaluating a probe

We'll fit one logistic-regression probe per layer and measure each with **cross-validated accuracy**, then pick the best layer.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# One probe per layer; report 5-fold cross-validated accuracy.
cv_acc = []
for L in range(n_layers):
    XL = store[:, L, :].astype(np.float32)
    clf = LogisticRegression(max_iter=1000, C=0.1)
    cv_acc.append(cross_val_score(clf, XL, labels, cv=5, scoring="accuracy").mean())

best_layer = int(np.argmax(cv_acc))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cv_acc, color="#3498db", lw=2, marker="o", ms=4)
ax.axvline(best_layer, color=DECEPTIVE_C, ls="--", lw=1.5,
           label=f"best layer {best_layer} ({cv_acc[best_layer]:.3f})")
ax.axhline(0.5, color="gray", ls=":", lw=1, label="chance")
ax.set_xlabel("layer"); ax.set_ylabel("probe accuracy (5-fold CV)")
ax.set_title("In-distribution linear probe accuracy by layer")
ax.set_ylim(0.4, 1.02); ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

# Fit the deployable probe on all data at the best layer.
probe = LogisticRegression(max_iter=1000, C=0.1).fit(store[:, best_layer, :].astype(np.float32), labels)
print(f"Best in-distribution layer: {best_layer}  (CV accuracy {cv_acc[best_layer]:.3f})")

#### The key idea that decides the competition: generalization

That ~0.9+ in-distribution accuracy is **not** your competition score, and chasing it will mislead you. Here's the trap:

- In-distribution accuracy is **near-saturated** — most layers score well, because within one scenario the probe can latch onto features specific to *that* scenario's prompts (its templates, topic, phrasing) rather than deception itself.
- The competition tests your probe on **held-out lies it never trained on** — different scenarios, and ultimately the red-team **imposter models**. A probe that aces one scenario routinely **collapses to near-chance (or worse — *inverted*, below 0.5)** on another.

So the number that matters is **transfer**: train on one distribution, evaluate on a *different* one. And critically — **never select your layer (or any hyperparameter) on the same data you report.** Use a held-out *validation* set to pick the layer, and a separate *test* set for the honest number.

Let's measure transfer directly: train on this scenario, evaluate on a different one.

In [ ]:
from sklearn.metrics import roc_auc_score

# Pick a DIFFERENT scenario (same captured model) as the transfer target.
scenarios = [p.split("scenario=")[-1].rstrip("/") for p in list_prefixes(f"data/captured_model={MODEL}/")]
TRANSFER_SCENARIO = next(s for s in scenarios if s != SCENARIO)
print(f"train on: {SCENARIO}\ntransfer to: {TRANSFER_SCENARIO}\n")

# Load the transfer scenario's metadata + activations.
tbase = f"captured_model={MODEL}/scenario={TRANSFER_SCENARIO}/split={SPLIT}"
df_t   = pd.read_parquet(f"s3://{BUCKET}/metadata/{tbase}/records.parquet", filesystem=fs)
y_t    = df_t["deceptive"].values
store_t = zarr.open(f"s3://{BUCKET}/data/{tbase}/activations.zarr", mode="r", storage_options={"anon": True})

# Per layer: train on scenario A (all rows), score AUROC in-distribution (CV) vs. on scenario B.
indist_auroc, transfer_auroc = [], []
for L in range(n_layers):
    XA = store[:, L, :].astype(np.float32)
    XB = store_t[:, L, :].astype(np.float32)
    clf = LogisticRegression(max_iter=1000, C=0.1).fit(XA, labels)
    indist_auroc.append(roc_auc_score(labels, clf.decision_function(XA)))   # optimistic (same data)
    transfer_auroc.append(roc_auc_score(y_t,   clf.decision_function(XB)))  # honest (unseen scenario)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(indist_auroc,   color="#3498db", lw=2, marker=".", label="in-distribution (train scenario)")
ax.plot(transfer_auroc, color=DECEPTIVE_C, lw=2, marker=".", label="transfer (held-out scenario)")
ax.axhline(0.5, color="gray", ls=":", lw=1, label="chance")
ax.set_xlabel("layer"); ax.set_ylabel("AUROC"); ax.set_ylim(0, 1.02)
ax.set_title("In-distribution vs. transfer: the gap is the real story")
ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print(f"best in-distribution AUROC: {max(indist_auroc):.3f}")
print(f"best transfer AUROC:        {max(transfer_auroc):.3f}  <- the number that reflects competition difficulty")

## Where to go from here

You now have the full loop: load data (§1.1–1.2), extract activations (§1.3), and train + **honestly evaluate** a probe (§2). The baseline linear probe is your starting point. Ideas to push on:

- **Train on more scenarios.** A probe trained on a single scenario overfits to it. Pool several Liars' Bench configs (and both labels balanced per config) so the probe sees *deception*, not one prompt template.
- **Pooling & position.** We mean-pooled the whole response. Try the last response token, max-pooling, or attaching the probe at a specific token.
- **Better detectors than logistic regression.** Difference-of-means directions, contrastive methods, or probes regularized to be robust across scenarios.
- **Cover all lie types.** Re-read the §1 taxonomy — a detector strong on *instructed* lies may miss *inherent* (backdoor) ones. Test across the axes deliberately.
- **Always select on validation, report on test.** It's the difference between a real result and a number that evaporates on the leaderboard.

### Resources
- [Liars' Bench paper](https://arxiv.org/abs/2511.16035) (Kretschmar et al., 2025) — the deception taxonomy and dataset construction
- [`Cadenza-Labs/liars-bench`](https://huggingface.co/datasets/Cadenza-Labs/liars-bench) — raw transcripts
- [nnsight docs](https://nnsight.net) and [NDIF](https://ndif.us) — remote activation extraction
- Get your NDIF API key at [login.ndif.us](https://login.ndif.us)

---
*Clear all cell outputs before committing or sharing this notebook.*